In [1]:
import nltk          # first I will import libraries
import pandas as pd
import numpy as np
import sklearn
import re

In [2]:
complaints = pd.read_csv("complaints_processed.csv")  # import the complaints file

In [3]:
complaints.head()   #check the first 5 lines

,Unnamed: 0,product,narrative
0,0,credit_card,purchase order day shipping amount receive pro...
1,1,credit_card,forwarded message date tue subject please inve...
2,2,retail_banking,forwarded message cc sent friday pdt subject f...
3,3,credit_reporting,payment history missing credit report speciali...
4,4,credit_reporting,payment history missing credit report made mis...


In [4]:
complaints.isnull().sum()   # check for missing values

Unnamed: 0     0
product        0
narrative     10
dtype: int64

In [5]:
complaints = complaints.dropna(subset=["narrative"]).copy()   # clean the missing values

In [6]:
complaints["narrative"] = complaints["narrative"].str.lower()  #  write everything in lowcase

In [7]:
def strip_noise(text):                      #clean the noise
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [8]:
from nltk.tokenize import word_tokenize      #  import tokenize from library
complaints["new_word_list"] = complaints["narrative"].apply(word_tokenize)  # add new_word_list column and paste tokenized words

In [11]:
from nltk.corpus import stopwords    # import stopwords from the library
stopword_list = set(stopwords.words('english'))


In [12]:
complaints.head()  # check if the new_word_list applied and words applied

,Unnamed: 0,product,narrative,new_word_list
0,0,credit_card,purchase order day shipping amount receive pro...,"[purchase, order, day, shipping, amount, recei..."
1,1,credit_card,forwarded message date tue subject please inve...,"[forwarded, message, date, tue, subject, pleas..."
2,2,retail_banking,forwarded message cc sent friday pdt subject f...,"[forwarded, message, cc, sent, friday, pdt, su..."
3,3,credit_reporting,payment history missing credit report speciali...,"[payment, history, missing, credit, report, sp..."
4,4,credit_reporting,payment history missing credit report made mis...,"[payment, history, missing, credit, report, ma..."


In [16]:
def remove_stopwords(word_list):
    clean_list = []                  # create empty list, check each word against stopword_list
    for word in word_list:
        if word not in stopword_list:   # if word is not a stopword, add to clean_list and return it
            clean_list.append(word)
    return clean_list

complaints["new_word_list"] = complaints["new_word_list"].apply(remove_stopwords)

In [18]:
from nltk.stem import WordNetLemmatizer   #import lemmatizer from the library
normalizer = WordNetLemmatizer()

In [20]:
def lemmatize_words(word_list):
    clean_list = []             #  create empty list, check each word, clean addings, add roots to the new_word_list
    for word in word_list:
        clean_list.append(normalizer.lemmatize(word))
    return clean_list

complaints["new_word_list"] = complaints["new_word_list"].apply(lemmatize_words)

In [22]:
complaints["cleaned_text"] = complaints["new_word_list"].str.join(" ")  # clean commas, turn the words into strings and add spaces

In [23]:
complaints.head()  #  check if the clean text column is added

,Unnamed: 0,product,narrative,new_word_list,cleaned_text
0,0,credit_card,purchase order day shipping amount receive pro...,"[purchase, order, day, shipping, amount, recei...",purchase order day shipping amount receive pro...
1,1,credit_card,forwarded message date tue subject please inve...,"[forwarded, message, date, tue, subject, pleas...",forwarded message date tue subject please inve...
2,2,retail_banking,forwarded message cc sent friday pdt subject f...,"[forwarded, message, cc, sent, friday, pdt, su...",forwarded message cc sent friday pdt subject f...
3,3,credit_reporting,payment history missing credit report speciali...,"[payment, history, missing, credit, report, sp...",payment history missing credit report speciali...
4,4,credit_reporting,payment history missing credit report made mis...,"[payment, history, missing, credit, report, ma...",payment history missing credit report made mis...


In [27]:
from sklearn.feature_extraction.text import CountVectorizer   #first we check BoW-import

In [29]:
bow_vectorizer = CountVectorizer(max_features=1000)
bow_matrix = bow_vectorizer.fit_transform(complaints["cleaned_text"])

In [31]:
bow_matrix.shape  # check how many words

(162411, 1000)

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer  #import vectorizer from library

In [36]:
tfidf_vectorizer = TfidfVectorizer(max_features=1000)      # calculate importance of each word
tfidf_matrix = tfidf_vectorizer.fit_transform(complaints["cleaned_text"])

In [38]:
tfidf_matrix.shape   #check 

(162411, 1000)

In [40]:
from sklearn.decomposition import LatentDirichletAllocation   #LDA runs with BoW matrix

In [41]:
lda_model = LatentDirichletAllocation(n_components=5, random_state=42)  # create LDA model with 5 topics 
# random_state=42 (Ultimate answer of the universe) ensures same results every run (reproducibility)
lda_output = lda_model.fit_transform(bow_matrix)  #  apply the model to BoW matrix

In [43]:
feature_names = bow_vectorizer.get_feature_names_out()  # show top 10 words for each LDA topic

for topic_index, topic in enumerate(lda_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-10:]]
    print(f"Topic {topic_index + 1}: {top_words}")

Topic 1: ['due', 'month', 'time', 'would', 'paid', 'mortgage', 'late', 'credit', 'loan', 'payment']
Topic 2: ['alleged', 'law', 'credit', 'provide', 'company', 'proof', 'reporting', 'claim', 'collection', 'debt']
Topic 3: ['theft', 'identity', 'section', 'report', 'agency', 'credit', 'reporting', 'consumer', 'information', 'account']
Topic 4: ['reporting', 'please', 'letter', 'remove', 'item', 'bureau', 'information', 'account', 'report', 'credit']
Topic 5: ['back', 'time', 'received', 'called', 'told', 'would', 'call', 'bank', 'card', 'account']


In [45]:
from sklearn.decomposition import NMF  # import from library

In [48]:
nmf_model = NMF(n_components=5, random_state=42)  # create NMF model with 5 topics, same universe answer
nmf_output = nmf_model.fit_transform(tfidf_matrix)   # apply model to TF-IDF matrix

In [49]:
feature_names_tfidf = tfidf_vectorizer.get_feature_names_out()  # show top 10 words for each NMF topic using TF-IDF features

for topic_index, topic in enumerate(nmf_model.components_):
    top_words = [feature_names_tfidf[i] for i in topic.argsort()[-10:]]
    print(f"Topic {topic_index + 1}: {top_words}")

Topic 1: ['item', 'removed', 'please', 'remove', 'bureau', 'information', 'reporting', 'inquiry', 'report', 'credit']
Topic 2: ['month', 'mortgage', 'time', 'told', 'would', 'late', 'card', 'bank', 'loan', 'payment']
Topic 3: ['open', 'charge', 'balance', 'please', 'victim', 'opened', 'fraudulent', 'theft', 'identity', 'account']
Topic 4: ['alleged', 'proof', 'creditor', 'original', 'consumer', 'reporting', 'agency', 'company', 'collection', 'debt']
Topic 5: ['investigation', 'inaccurate', 'information', 'received', 'sent', 'response', 'item', 'letter', 'dispute', 'day']
